# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Rows are tutoring sessions; students appear in many rows and there is no date column.

Design decisions (from `checklist.md` and `columns.csv`):
- **Entity split:** the test set and every CV fold hold *whole students* (`student_key`), so the model is judged on students it has not seen.
- **No time column exists**, so a temporal split is impossible; grouping by student is the applicable structure.
- **Identifier dropped:** `student_key` is only the grouping variable, never a feature.
- **Features known at prediction time.** The quiz *closes* the session, so `session_minutes` (length of the whole session) is only final once the quiz is over; it is excluded. `hints_used` is counted while the student works, before the quiz, so it is kept (assumption: prediction is made just before the quiz starts). `prior_score` (placement test), `enrol_year`, `school_code` are known beforehand.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, brier_score_loss
from sklearn.base import clone
SEED = 0
df = pd.read_csv('data.csv')
print(df.shape, df.passed_quiz.mean())
print(df.groupby('student_key').size().describe())

## 1. Hold out the test set (whole students) before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), train.passed_quiz.mean(), test.passed_quiz.mean())
# The test set is not touched again until the final scoring cell.

## 2. Candidates, named before scoring anything
1. `baseline`: DummyClassifier (class prior); AUC 0.5 by construction
2. `logit`: logistic regression on `prior_score, hints_used, enrol_year` (standardised)
3. `logit_school`: as `logit` plus one-hot `school_code`
4. `gbm`: HistGradientBoosting on `prior_score, hints_used, enrol_year`
5. `logit_prior_only`: logistic regression on `prior_score` alone

Selection metric: ROC AUC over grouped CV folds. Hyperparameters are fixed defaults (no tuning). The decision threshold is chosen afterwards from the chosen model's out-of-fold predictions by maximising balanced accuracy.

In [ ]:
NUM = ['prior_score', 'hints_used', 'enrol_year']
def lr(cols, cat=()):
    tr = [('n', StandardScaler(), list(cols))]
    if cat: tr.append(('c', OneHotEncoder(handle_unknown='ignore'), list(cat)))
    return Pipeline([('pre', ColumnTransformer(tr)), ('m', LogisticRegression(max_iter=1000))])
CANDS = {
 'baseline': (Pipeline([('pre', ColumnTransformer([('n', 'passthrough', NUM)])), ('m', DummyClassifier(strategy='prior'))]), NUM),
 'logit': (lr(NUM), NUM),
 'logit_school': (lr(NUM, ['school_code']), NUM + ['school_code']),
 'gbm': (Pipeline([('pre', ColumnTransformer([('n', 'passthrough', NUM)])),
                   ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED))]), NUM),
 'logit_prior_only': (lr(['prior_score']), ['prior_score']),
}

## 3. Grouped cross-validation on training rows only
Scaling and encoding sit inside each pipeline, so they are fit on the training part of each fold only.

In [ ]:
folds = list(GroupKFold(n_splits=5).split(train, groups=train.student_key))
y = train.passed_quiz.values
oof = {}; rows = []
for name, (pipe, cols) in CANDS.items():
    p = np.zeros(len(train)); aucs = []
    for a, b in folds:
        m = clone(pipe).fit(train.iloc[a][cols], y[a])
        p[b] = m.predict_proba(train.iloc[b][cols])[:, 1]
        aucs.append(roc_auc_score(y[b], p[b]))
    oof[name] = p
    rows.append(dict(model=name, auc_mean=np.mean(aucs), auc_sd=np.std(aucs), brier=brier_score_loss(y, p)))
cv = pd.DataFrame(rows).set_index('model').round(4)
cv

## 4. Choose model and threshold on CV only

In [ ]:
best = cv.drop('baseline').auc_mean.idxmax(); print('chosen:', best)
ths = np.linspace(0.05, 0.95, 91)
bal = [balanced_accuracy_score(y, oof[best] >= t) for t in ths]
THR = float(ths[int(np.argmax(bal))])
print('threshold', THR, 'CV balanced acc', round(max(bal), 3))
print('CV balanced acc at 0.5:', round(balanced_accuracy_score(y, oof[best] >= 0.5), 3))

**Recorded choice (fixed before the test set is scored):** the model and threshold printed above, selected only from grouped CV on training students.

## 5. Refit on all training rows; score the test set once

In [ ]:
pipe, cols = CANDS[best]
final = clone(pipe).fit(train[cols], y)
pt = final.predict_proba(test[cols])[:, 1]; yt = test.passed_quiz.values
auc = roc_auc_score(yt, pt)
# students are clustered, so bootstrap over students for the interval
rng = np.random.default_rng(SEED); g = test.student_key.values; ids = np.unique(g)
idx = {s: np.where(g == s)[0] for s in ids}; bs = []
for _ in range(1000):
    ii = np.concatenate([idx[s] for s in rng.choice(ids, len(ids))])
    if len(np.unique(yt[ii])) > 1: bs.append(roc_auc_score(yt[ii], pt[ii]))
print(f'TEST AUC {auc:.3f}  (95% student-bootstrap CI {np.percentile(bs, 2.5):.3f}-{np.percentile(bs, 97.5):.3f})')
print('balanced acc @thr', round(balanced_accuracy_score(yt, pt >= THR), 3), ' accuracy', round(accuracy_score(yt, pt >= THR), 3),
      ' majority-class accuracy', round(max(yt.mean(), 1 - yt.mean()), 3))

## 6. Interpretation
The headline is the single test AUC above against 0.5 for the trivial baseline. The CV table shows whether school codes, nonlinearity or extra features beat the simpler models; the test set played no part in those comparisons.

Caveats: no time column, so generalisation to *later* sessions is untested; `hints_used` assumes prediction at quiz start; `session_minutes` was excluded on timing grounds and might add signal if it were known in advance; one test split of a few hundred students gives a wide interval.